# Iniciación de paquetes

In [2]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np

# Métricas iniciales

In [134]:
import math

def metricas_estaticas(clientes, horas, mu, s, cs=0, cq=0):
   
    lambda_tasa = clientes / horas
    rho = lambda_tasa / (s * mu)
    

    if rho >= 1:
        return {"error": "Sistema inestable (rho >= 1)"}


    termino_sumatorio = sum([(lambda_tasa/mu)**n / math.factorial(n) for n in range(s)])
    termino_cola = ((lambda_tasa/mu)**s / (math.factorial(s) * (1 - rho)))
    pi0 = 1 / (termino_sumatorio + termino_cola)
    
  
    Lq = (pi0 * ((lambda_tasa/mu)**s) * rho) / (math.factorial(s) * (1 - rho)**2)
    Wq = Lq / lambda_tasa
    W = Wq + (1/mu)
    L = lambda_tasa * W
    
    
    Cts = (cs * s) + (cq * Lq)
    
    # Devolvemos un diccionario con todo
    return {
        "rho": round(rho, 3),
        "Lq": round(Lq, 3),
        "Wq_min": round(Wq*60, 3),
        "L": round(L, 3),
        "Cts": round(Cts, 3),
        "lambda": round(lambda_tasa, 3),
        "horas": horas,
        "clientes": clientes,
        "mu": mu,
        "servidores": s
    }

Aún no nos interesa la optimización del dinero, de momento vamos a fijarnos en los factores de utilización y ocupacion de las colas

In [223]:
metricas_teoricas = metricas_estaticas(120, 8, 3, 7)
print(metricas_teoricas)

{'rho': 0.714, 'Lq': 0.81, 'Wq_min': 3.241, 'L': 5.81, 'Cts': 0.0, 'lambda': 15.0, 'horas': 8, 'clientes': 120, 'mu': 3, 'servidores': 7}


In [225]:
metricas_teoricas6 = metricas_estaticas(120, 8, 3, 6)
metricas_teoricas5 = metricas_estaticas(120, 8, 3, 5)
print(metricas_teoricas6)
print(metricas_teoricas5)

{'rho': 0.833, 'Lq': 2.938, 'Wq_min': 11.75, 'L': 7.938, 'Cts': 0.0, 'lambda': 15.0, 'horas': 8, 'clientes': 120, 'mu': 3, 'servidores': 6}
{'error': 'Sistema inestable (rho >= 1)'}


# Simulación de las llegadas

In [137]:
pesos_lambda = [1.3, 1.2, 0.9, 0.7, 0.8, 1.2, 1.1, 0.8]

tasa_hora = []
for peso in pesos_lambda:
    lambda_hora = metricas_teoricas["lambda"] * peso
    tasa_hora.append(lambda_hora)

clientes_por_Hora = []
horas_llegada = []
for hora in range(len(tasa_hora)):
    n_clientes = np.random.poisson(tasa_hora[hora])
    clientes_por_Hora.append(n_clientes)
    for cliente in range(n_clientes):
        hora_llegada = np.random.uniform(hora, hora+1)
        horas_llegada.append(hora_llegada)

horas_llegada.sort()

clientes_T = sum(clientes_por_Hora)
print(clientes_T)
print(horas_llegada)

144
[0.0006983677144767331, 0.07064156723767367, 0.09719107036910446, 0.12019645645636523, 0.1274930327902456, 0.24963037395725807, 0.25745244830840686, 0.26296252740409476, 0.27922649331675753, 0.35012630273527223, 0.4253742349896136, 0.42580047736537807, 0.4326499784918385, 0.4983075277585248, 0.5980310171814487, 0.6754841828462896, 0.6979103760142884, 0.7500677059070602, 0.7659864930825379, 0.7839331450458006, 0.7876267193402663, 0.79700380171067, 0.9008441753718262, 0.9039066217246392, 0.9203335862819881, 1.0792162274462305, 1.1160830429422695, 1.1483825597957464, 1.1659117811957764, 1.1802788336024708, 1.2009887886812678, 1.2965218453122207, 1.3734140673686508, 1.3910022757338258, 1.4160900212095746, 1.418212894184678, 1.4349073039374052, 1.5274764495438666, 1.5814358968092992, 1.6281724906415667, 1.652629903924356, 1.659322222290212, 1.6917499509059724, 1.7080313094715278, 1.7519585751320919, 1.8238349857031815, 1.8784130555055354, 1.8956939646499824, 1.9659114405368667, 2.027136

# Salidas del sistema y tiempos de atención

In [ ]:
# 1. Definir número de asesores
n_asesores = metricas_teoricas['servidores'] 
# 2. LISTA de disponibilidad (un valor por cada asesor)
disponibilidad_asesores = [0.0] * n_asesores 

esperas_dia = []

for llegada in horas_llegada:
    
    momento_asesor_libre = min(disponibilidad_asesores)
    indice = disponibilidad_asesores.index(momento_asesor_libre)
    
    inicio_atencion = max(llegada, momento_asesor_libre)
    
    espera_w_q = inicio_atencion - llegada
    esperas_dia.append(espera_w_q)
    
    t_servicio = np.random.exponential(1/metricas_teoricas['mu'])
    disponibilidad_asesores[indice] = inicio_atencion + t_servicio

print(esperas_dia)
print(len(esperas_dia))

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.04817502259052325, 0.05110154505536335, 0.00030610454693252853, 0.0, 0.08160039310751444, 0.14155603873480604, 0.0921452853256719, 0.09060005521856584, 0.07610053939485217, 0.11092218313421409, 0.12220078157794867, 0.05261474539468369, 0.07608190500388823, 0.21491733634935972, 0.08184906360747424, 0.08137227809719771, 0.07350747487738829, 0.16121913070477278, 0.20442027691754494, 0.19103340042896977, 0.13484455713793064, 0.13377651969239857, 0.12772270086554482, 0.11185736198878593, 0.12685957119561997, 0.12366288661262304, 0.1022981039448636, 0.19316100536271374, 0.17673310117563767, 0.18814440919387065, 0.22511440630375823, 0.23984222016732093, 0.32195506105566474, 0.32409414949768456, 0.39360949365800746, 0.3598088488890383, 0.5723468294810556, 0.5071602402391542, 0.5009578125839953, 0.4980047112630994, 0.4688215488338159, 0.5009530588107043, 0.2800602530895393, 0.2986628601226835, 0.2700342431953717, 0.19281075953827687, 0.3

In [210]:
Wq_simulado = round(np.mean(esperas_dia)*60, 4)
print(Wq_simulado, metricas_teoricas['Wq_min'])

1.6307 12.277
